# Bài 3: Kỹ Thuật Xử Lý Dữ Liệu Hiệu Quả với LightGBM

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu và tận dụng khả năng xử lý biến categorical tự nhiên của LightGBM.
- So sánh ưu và nhược điểm của việc sử dụng tính năng này so với các phương pháp truyền thống như One-Hot Encoding và Label Encoding.
- Biết cách xử lý giá trị thiếu (missing values) một cách hiệu quả trong LightGBM.
- Áp dụng các kỹ thuật này để đơn giản hóa quy trình tiền xử lý và cải thiện hiệu suất mô hình.

## Tại sao xử lý dữ liệu lại quan trọng cho LightGBM?

Garbage in, garbage out (Rác vào, rác ra). Chất lượng của mô hình phụ thuộc rất nhiều vào chất lượng của dữ liệu đầu vào. Tuy nhiên, một trong những ưu điểm lớn của các mô hình dựa trên cây quyết định như LightGBM là chúng ít nhạy cảm với việc co giãn đặc trưng (feature scaling) và có thể xử lý một số loại dữ liệu phi tuyến tính một cách tự nhiên.

Đặc biệt, LightGBM có những cơ chế tích hợp sẵn rất mạnh mẽ để xử lý hai trong số những vấn đề phổ biến nhất trong tiền xử lý dữ liệu: **biến categorical** và **giá trị thiếu**. Hiểu và sử dụng đúng cách các tính năng này không chỉ giúp mô hình chạy nhanh hơn mà còn có thể mang lại kết quả tốt hơn.

## 1. Xử Lý Biến Categorical: Sức mạnh tiềm ẩn của LightGBM

Hầu hết các thuật toán Machine Learning yêu cầu tất cả các đầu vào phải là số. Điều này buộc chúng ta phải chuyển đổi các biến categorical (như 'thành phố', 'giới tính', 'loại sản phẩm') sang dạng số. Các phương pháp phổ biến bao gồm:

- **Label Encoding:** Ánh xạ mỗi giá trị duy nhất của một biến categorical sang một số nguyên (ví dụ: 'Hà Nội' -> 0, 'TP.HCM' -> 1, 'Đà Nẵng' -> 2). 
  - *Nhược điểm:* Mô hình có thể ngầm hiểu sai rằng có một mối quan hệ thứ tự giữa các giá trị (ví dụ: Đà Nẵng > TP.HCM > Hà Nội), điều này thường không đúng.

- **One-Hot Encoding (OHE):** Tạo một cột nhị phân mới cho mỗi giá trị duy nhất. 
  - *Nhược điểm:* Nếu một biến có nhiều giá trị duy nhất (cardinality cao), OHE sẽ tạo ra một ma trận đặc trưng khổng lồ và thưa, làm tăng đáng kể chi phí tính toán và bộ nhớ.

### Cách tiếp cận của LightGBM

LightGBM cung cấp một giải pháp rất hiệu quả: **xử lý trực tiếp các biến categorical**. Thay vì bạn phải thực hiện encoding trước, bạn chỉ cần chỉ định cho LightGBM biết cột nào là categorical.

**Cách hoạt động:**
Khi gặp một biến categorical, thay vì tìm điểm chia trên các giá trị số, LightGBM sử dụng một thuật toán đặc biệt (dựa trên phương pháp của Fisher) để tìm cách phân chia tối ưu các category thành hai nhóm. Nó sẽ duyệt qua các cách nhóm các category lại với nhau (ví dụ: nhóm {Hà Nội, Đà Nẵng} và {TP.HCM}) và chọn cách nhóm nào mang lại gain lớn nhất. Điều này hiệu quả hơn nhiều so với việc tạo ra hàng chục, hàng trăm cột từ OHE.

**Lợi ích:**
- **Tốc độ:** Nhanh hơn đáng kể so với việc sử dụng dữ liệu đã được One-Hot-Encode, đặc biệt với các biến có cardinality cao.
- **Hiệu suất:** Thường cho kết quả tốt hơn hoặc tương đương, vì nó tránh được "lời nguyền số chiều" (curse of dimensionality) và cho phép cây quyết định đưa ra các quyết định phức tạp hơn trên các biến categorical.
- **Đơn giản:** Giảm thiểu bước tiền xử lý dữ liệu.

### Ví dụ: So sánh Label Encoding, OHE và xử lý tự nhiên

Chúng ta sẽ sử dụng một bộ dữ liệu về giá xe hơi để minh họa.

In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import time

# Tải dữ liệu
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
cols = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 
        'drive-wheels', 'engine-location', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 
        'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke', 
        'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']
df = pd.read_csv(url, header=None, names=cols, na_values='?')

# Xử lý cơ bản
df.dropna(subset=['price'], inplace=True)
for col in ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

# Xác định các cột categorical
categorical_features = [col for col in df.columns if df[col].dtype == 'object']
print(f"Các cột Categorical: {categorical_features}")

#### Cách 1: One-Hot Encoding

In [ ]:
df_ohe = pd.get_dummies(df, columns=categorical_features, drop_first=True)

X_ohe = df_ohe.drop('price', axis=1)
y_ohe = df_ohe['price']

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(X_ohe, y_ohe, test_size=0.2, random_state=42)

start_time = time.time()
lgbm_ohe = lgb.LGBMRegressor(random_state=42)
lgbm_ohe.fit(X_train_ohe, y_train_ohe)
y_pred_ohe = lgbm_ohe.predict(X_test_ohe)
rmse_ohe = np.sqrt(mean_squared_error(y_test_ohe, y_pred_ohe))
duration_ohe = time.time() - start_time

print("--- One-Hot Encoding ---")
print(f"Số lượng features: {X_train_ohe.shape[1]}")
print(f"RMSE: {rmse_ohe:.2f}")
print(f"Thời gian huấn luyện: {duration_ohe:.4f} giây")

#### Cách 2: Xử lý tự nhiên của LightGBM

Để sử dụng tính năng này, chúng ta chỉ cần chuyển đổi kiểu dữ liệu của các cột categorical sang `category` trong Pandas. LightGBM sẽ tự động nhận diện và xử lý chúng.

In [ ]:
df_native = df.copy()
# Xử lý giá trị thiếu trong các cột categorical trước khi chuyển đổi
for col in categorical_features:
    df_native[col].fillna('missing', inplace=True)
    df_native[col] = df_native[col].astype('category')

X_native = df_native.drop('price', axis=1)
y_native = df_native['price']

X_train_native, X_test_native, y_train_native, y_test_native = train_test_split(X_native, y_native, test_size=0.2, random_state=42)

start_time = time.time()
lgbm_native = lgb.LGBMRegressor(random_state=42)
# Không cần truyền categorical_feature='auto' vì kiểu dữ liệu đã là 'category'
lgbm_native.fit(X_train_native, y_train_native)
y_pred_native = lgbm_native.predict(X_test_native)
rmse_native = np.sqrt(mean_squared_error(y_test_native, y_pred_native))
duration_native = time.time() - start_time

print("\n--- Xử lý tự nhiên của LightGBM ---")
print(f"Số lượng features: {X_train_native.shape[1]}")
print(f"RMSE: {rmse_native:.2f}")
print(f"Thời gian huấn luyện: {duration_native:.4f} giây")

**Phân tích kết quả:**
Bạn sẽ thấy rằng cách xử lý tự nhiên của LightGBM:
- Giữ nguyên số lượng features, trong khi OHE làm số features tăng vọt.
- Thời gian huấn luyện nhanh hơn đáng kể.
- Kết quả RMSE (Root Mean Squared Error) thường tốt hơn hoặc tương đương.

**=> Quy tắc chung:** Luôn ưu tiên sử dụng cách xử lý categorical tự nhiên của LightGBM. Chỉ sử dụng Label/One-Hot Encoding nếu bạn có lý do đặc biệt hoặc khi sử dụng các mô hình khác không hỗ trợ tính năng này.

## 2. Xử Lý Giá Trị Thiếu (Missing Values)

Một ưu điểm tuyệt vời khác của LightGBM (và các mô hình cây nói chung) là khả năng **xử lý các giá trị thiếu một cách tự nhiên**.

**Cách hoạt động:**
Khi tìm kiếm điểm chia tốt nhất cho một feature, nếu gặp các giá trị `NaN`, LightGBM sẽ không báo lỗi. Thay vào đó, nó sẽ thử hai kịch bản:
1.  Gán tất cả các hàng có giá trị thiếu vào nhánh bên trái.
2.  Gán tất cả các hàng có giá trị thiếu vào nhánh bên phải.

Sau đó, nó sẽ tính toán information gain cho cả hai kịch bản và chọn kịch bản nào tốt hơn. Điều này có nghĩa là mô hình tự học cách xử lý giá trị thiếu một cách tối ưu nhất dựa trên dữ liệu. Trong nhiều trường hợp, bản thân việc một giá trị bị thiếu đã là một thông tin hữu ích, và LightGBM có thể khai thác được điều đó.

**Lợi ích:**
- **Không cần Imputation:** Bạn không cần phải thực hiện các bước phức tạp như điền giá trị thiếu bằng trung bình, trung vị, hay một giá trị cố định (mean/median/constant imputation).
- **Bảo toàn thông tin:** Giữ lại được thông tin tiềm ẩn trong việc giá trị bị thiếu.

**Lưu ý:**
- LightGBM mặc định coi `np.nan` là giá trị thiếu.
- Nếu dữ liệu của bạn sử dụng một ký hiệu khác (ví dụ: '?', -999), bạn cần chuyển đổi chúng thành `np.nan` trước.
- Đối với các biến categorical, bạn nên điền giá trị thiếu bằng một category riêng (ví dụ: 'missing') trước khi chuyển đổi kiểu dữ liệu, như đã làm ở ví dụ trên. Điều này cho phép LightGBM coi 'missing' như một category bình thường.

## Tóm tắt

- LightGBM có khả năng xử lý **biến categorical** một cách tự nhiên và hiệu quả. Hãy ưu tiên sử dụng tính năng này bằng cách chuyển đổi kiểu dữ liệu của cột sang `'category'` trong Pandas.
- Kỹ thuật này thường nhanh hơn và cho kết quả tốt hơn so với One-Hot Encoding, đặc biệt với các biến có cardinality cao.
- LightGBM cũng có thể xử lý **giá trị thiếu (missing values)** trong các biến số một cách tự động. Nó tự học cách phân bổ các giá trị `NaN` vào nhánh trái hoặc phải để tối đa hóa gain.
- Việc tận dụng các tính năng này giúp đơn giản hóa đáng kể quy trình tiền xử lý dữ liệu và giúp bạn tập trung hơn vào việc tinh chỉnh mô hình.

## Bài học tiếp theo

Chúng ta đã biết cách xây dựng, tinh chỉnh và chuẩn bị dữ liệu cho mô hình. Nhưng làm thế nào để chắc chắn rằng các tham số chúng ta chọn là tốt nhất? Trong bài học tiếp theo, chúng ta sẽ tìm hiểu về các kỹ thuật kiểm định chéo (Cross-Validation) và các phương pháp tự động để tìm kiếm không gian tham số (Hyperparameter Tuning).